# Notebook 7: Validation & Sanity Checks
**Paper 5 – Cross-Framework Quantum Algorithm Benchmarking**

**Purpose**: Validate all pipeline outputs for correctness before paper writing:

1. **Algorithm validation**: Verify each algorithm produces the expected output distribution
2. **QASM validation**: Check QASM files parse correctly; no malformed syntax
3. **Statistical validation**: Verify JSD scores are within expected ranges
4. **QPack calibration**: Compare S_overall scores to published QPack reference values
5. **Reproducibility check**: Verify results are stable across 10 simulation repeats

**Prerequisites**: All notebooks 1–6 must be complete.

**Pass/Fail criteria documented inline** — note any failures for the paper's Threats to Validity section.

In [ ]:
import os, sys, json, warnings
QCANVAS_ROOT = os.path.abspath('../..')
if QCANVAS_ROOT not in sys.path:
    sys.path.insert(0, QCANVAS_ROOT)
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

PASS = '✓ PASS'
FAIL = '✗ FAIL'
WARN = '⚠  WARN'

def check(condition, label, detail=''):
    status = PASS if condition else FAIL
    print(f'  {status}  {label}')
    if detail:
        print(f'         {detail}')
    return condition

print('Validation Suite — Paper 5')
print('=' * 60)

In [ ]:
# ── 1. File existence checks ─────────────────────────────────────────────────
print('\n[1] Required output files:')
required_files = [
    'benchmarks/metrics/compilation_times.csv',
    'benchmarks/metrics/structural_metrics.csv',
    'benchmarks/metrics/simulation_metrics.csv',
    'benchmarks/metrics/quantum_volume_estimates.csv',
    'benchmarks/metrics/statistical_tests.csv',
    'benchmarks/metrics/qpack_scores.csv',
    'benchmarks/metrics/power_law_fits.csv',
]
all_files_ok = all(check(os.path.exists(f), f) for f in required_files)

In [ ]:
# ── 2. QASM output count check ───────────────────────────────────────────────
print('\n[2] QASM output files:')
qasm_dir   = 'benchmarks/qasm_outputs'
qasm_files = [f for f in os.listdir(qasm_dir) if f.endswith('.qasm')] if os.path.isdir(qasm_dir) else []
# Expected: 12 algos × 3 frameworks × (variable qubit counts)
# Minimum 12 × 3 = 36 (if only one qubit count each)
check(len(qasm_files) >= 36, f'{len(qasm_files)} QASM files generated (expected ≥ 36)')

In [ ]:
# ── 3. Bell State distribution validation ────────────────────────────────────
print('\n[3] Bell State distribution validation:')
dist_dir = 'benchmarks/metrics/distributions'

if os.path.isdir(dist_dir):
    for fw in ['qiskit', 'cirq', 'pennylane']:
        path = os.path.join(dist_dir, f'bell_state_2q_{fw}_4096_trial1.json')
        if os.path.exists(path):
            with open(path) as f:
                counts = json.load(f)
            total   = sum(counts.values())
            p_00    = counts.get('00', 0) / total
            p_11    = counts.get('11', 0) / total
            # Bell state: expect P(|00⟩) ≈ 0.5, P(|11⟩) ≈ 0.5, P(other) ≈ 0
            p_other = 1 - p_00 - p_11
            check(abs(p_00 - 0.5) < 0.05 and abs(p_11 - 0.5) < 0.05,
                  f'Bell State {fw}: P(|00⟩)={p_00:.3f}, P(|11⟩)={p_11:.3f}',
                  f'P(other)={p_other:.3f} (should ≈ 0)')
        else:
            print(f'  {WARN}  {fw}: distribution file not found')

In [ ]:
# ── 4. JSD range validation ──────────────────────────────────────────────────
print('\n[4] JSD range validation:')
if os.path.exists('benchmarks/metrics/statistical_tests.csv'):
    df_t = pd.read_csv('benchmarks/metrics/statistical_tests.csv')
    jsd_cols = ['jsd_qiskit_cirq', 'jsd_qiskit_pl', 'jsd_cirq_pl']
    for col in jsd_cols:
        if col in df_t.columns:
            vals = df_t[col].dropna()
            in_range = (vals >= 0).all() and (vals <= 1).all()
            max_val  = vals.max()
            mean_val = vals.mean()
            check(in_range, f'{col}: all values in [0,1]',
                  f'max={max_val:.4f}, mean={mean_val:.4f}')
            if max_val > 0.05:
                divergent_algos = df_t[df_t[col] > 0.05]['algorithm'].tolist()
                print(f'    {WARN}  Divergent algorithms: {divergent_algos}')

In [ ]:
# ── 5. QPack score calibration ───────────────────────────────────────────────
print('\n[5] QPack S_overall calibration:')
if os.path.exists('benchmarks/metrics/qpack_scores.csv'):
    df_q = pd.read_csv('benchmarks/metrics/qpack_scores.csv')
    print('  Published QPack reference values (Donkers et al. 2022):')
    print('    QuEST=183.2, Cirq Sim=157.6, Qiskit Aer=147.2, Rigetti=104.8')
    print('  Our QSim (noiseless Cirq backend) results:')
    for _, row in df_q.iterrows():
        s_overall = row.get('S_overall', float('nan'))
        if not np.isnan(s_overall):
            in_expected_range = 50 < s_overall < 400
            check(in_expected_range,
                  f"{row['framework']:12s}: S_overall={s_overall:.2f}",
                  '(expected 50–400 for noiseless statevector simulator)')

In [ ]:
# ── 6. Reproducibility: simulation time CV ───────────────────────────────────
print('\n[6] Simulation reproducibility (coefficient of variation < 20%):')
if os.path.exists('benchmarks/metrics/simulation_metrics.csv'):
    df_s = pd.read_csv('benchmarks/metrics/simulation_metrics.csv')
    df_s_4k = df_s[(df_s['shots'] == 4096) & df_s['success']]
    for fw in ['qiskit', 'cirq', 'pennylane']:
        sub = df_s_4k[df_s_4k['framework'] == fw]
        if not sub.empty:
            mean_t = sub['mean_sim_time_ms'].mean()
            std_t  = sub['std_sim_time_ms'].mean()
            cv     = (std_t / mean_t) * 100 if mean_t > 0 else float('nan')
            check(cv < 20, f'{fw:12s}: CV={cv:.1f}% (std/mean of sim times)',
                  f'mean={mean_t:.1f} ms, std={std_t:.1f} ms')

In [ ]:
# ── 7. Power-law fit quality (R² > 0.85) ────────────────────────────────────
print('\n[7] Power-law fit quality (R² > 0.85):')
if os.path.exists('benchmarks/metrics/power_law_fits.csv'):
    df_pl = pd.read_csv('benchmarks/metrics/power_law_fits.csv')
    for _, row in df_pl.iterrows():
        r2 = row.get('r2', float('nan'))
        if not np.isnan(r2):
            check(r2 > 0.85, f"{row['framework']:12s} | {row['algorithm']:25s}: R²={r2:.3f}, a={row['a']:.3f}")

print('\n' + '=' * 60)
print('Validation complete. Review any failures above before paper submission.')